In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Calibration Resources Guide

Orientation guide to CUDA-Q Academic **Calibration** resources: bring-up visualizations, sample plots, and ways to run [NVIDIA Ising Calibration](https://www.nvidia.com/en-us/solutions/quantum-computing/ising/). Use this notebook to choose a path; use the [Ising Calibration Introductory Demo](ising_calibration_intro.ipynb) for the full prompting and analysis walkthrough. Hardware access is not required.

→ [README.md](README.md) · [ising_calibration_intro.ipynb](ising_calibration_intro.ipynb) · [Calibration track](https://nvidia.github.io/cuda-q-academic/learningpath.html?track=track-calibration) · [Visualization Gallery](https://nvidia.github.io/cuda-q-academic/visualization-gallery.html)


## What you can learn

* The **order** of common single-qubit bring-up experiments (resonator → spectroscopy → Rabi → coherence → readout → benchmarking → pulse shaping)
* One physical insight per experiment via the [visualization gallery](https://nvidia.github.io/cuda-q-academic/visualization-gallery.html)
* How to **run Ising** (browser playground, API-key setup, intro notebook, or the upload widget below)

**Prerequisites:** comfort with |0⟩ / |1⟩, basic gates, and reading a scientific plot.

## Suggested exploration path

1. **Orient** — pick a bring-up experiment and note what it measures  
2. **Explore** — open the matching tool in the [visualization gallery](https://nvidia.github.io/cuda-q-academic/visualization-gallery.html)  
3. **Recognize** — compare with a real or [QCalEval](https://huggingface.co/datasets/nvidia/QCalEval) plot  
4. **Choose a setup** — [NIM playground](https://build.nvidia.com/nvidia/ising-calibration-1-35b-a3b) (browser) or [API-key setup](https://nvidia.github.io/cuda-q-academic/interactive_widgets/ising_api_key_setup.html)  
5. **Try Ising** — use the playground, the upload widget below, or continue to the [Ising Calibration Introductory Demo](ising_calibration_intro.ipynb) for zero-shot helpers and an ICL Ramsey comparison


## Running Ising

| Path | Needs | Best for |
|------|--------|----------|
| [NIM playground](https://build.nvidia.com/nvidia/ising-calibration-1-35b-a3b) | Browser only | Fastest start with no local setup |
| [Environment and API-key setup](https://nvidia.github.io/cuda-q-academic/interactive_widgets/ising_api_key_setup.html) | Free NVIDIA API key + local Python/Jupyter | Reusable local environment: venv → kernel → `NVIDIA_API_KEY` |
| [Ising Calibration Introductory Demo](ising_calibration_intro.ipynb) | API key + `openai` + extracted `calibration_images/` | Guided zero-shot and ICL workflow with sample plots |
| Cells below | API key + `requests` / `ipywidgets` | Colab or Jupyter upload-and-run |

**API keys:** create or manage via [NVIDIA Build](https://build.nvidia.com/). Never hard-code, print, save, or commit a key.

**Sample plots:** a few [QCalEval](https://huggingface.co/datasets/nvidia/QCalEval) PNGs suffice for a quick trial. For the intro notebook, extract [`images/calibration_images.zip`](images/calibration_images.zip) as described in [README.md](README.md).

Treat Ising output as a starting point for review; keep observations distinct from inferences (full patterns in the [Intro notebook](ising_calibration_intro.ipynb)).


## Notebook API (upload and run)

Lightweight try-it path: install if needed, edit the short prompt, upload one PNG/JPEG, paste your API key, and select **Run Ising**. Model: `nvidia/ising-calibration-1.5-31b`.

In [ ]:
# Install if needed
%pip install -q requests ipywidgets

In [ ]:
import base64
import requests
import ipywidgets as widgets
from IPython.display import display

# Auto-detect environment (Colab FileUpload layout differs from JupyterLab)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Short starter prompt — full patterns are in ising_calibration_intro.ipynb
PROMPT = """This is a quantum-calibration experiment plot.
Describe axes and key visible features; assess data vs fit separately;
extract only supported parameters (or null); recommend one bounded next step.
Keep observations distinct from inferences."""


def run_api(prompt, api_key, mime_type, image_b64):
    payload = {
        "model": "nvidia/ising-calibration-1.5-31b",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{mime_type};base64,{image_b64}"},
                },
            ],
        }],
        "temperature": 0.2,
        "max_tokens": 8192,
        "stream": False,
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=180,
    )
    if not response.ok:
        print(f"HTTP {response.status_code}")
        print(response.text[:2000])
        response.raise_for_status()
    print(response.json()["choices"][0]["message"]["content"])


prompt_box = widgets.Textarea(
    value=PROMPT,
    layout=widgets.Layout(width="100%", height="120px"),
)
uploader = widgets.FileUpload(accept="image/*", multiple=False)
api_key_box = widgets.Password(placeholder="Enter your NVIDIA API key")
run_button = widgets.Button(description="Run Ising", button_style="success")
out = widgets.Output()


def on_click(b):
    with out:
        out.clear_output()
        if not uploader.value:
            print("Please upload an image first.")
            return
        if not api_key_box.value:
            print("Please enter your API key.")
            return
        print("Calling Ising...")
        if IN_COLAB:
            uploaded_file = next(iter(uploader.value.values()))
            mime_type = uploaded_file["metadata"]["type"] or "image/png"
            image_b64 = base64.b64encode(uploaded_file["content"]).decode("utf-8")
        else:
            uploaded_file = uploader.value[0]
            mime_type = uploaded_file["type"] or "image/png"
            image_b64 = base64.b64encode(uploaded_file["content"]).decode("utf-8")
        run_api(prompt_box.value, api_key_box.value, mime_type, image_b64)


run_button.on_click(on_click)

display(
    widgets.Label("1. Edit your prompt:"),
    prompt_box,
    widgets.Label("2. Upload image:"),
    uploader,
    widgets.Label("3. Enter API key:"),
    api_key_box,
    run_button,
    out,
)

## Prompting checklist

* Axes / units / visible features · data quality vs fit · supported parameters only · one bounded next step · `null` when unknown  
* Verify numbers against the plot before acting  
* Full prompting patterns (`generic_analysis_task`, zero-shot, ICL): [ising_calibration_intro.ipynb](ising_calibration_intro.ipynb)

### Troubleshooting

* **401 / 403:** confirm the API key is valid and has not expired  
* **Model not found:** check the identifier in [NVIDIA API docs](https://docs.api.nvidia.com/nim/reference/nvidia-ising-calibration-1-35b-a3b) or NVIDIA Build  
* **Widget does not render:** rerun install → restart kernel → rerun interface; local kernels: [setup guide](https://nvidia.github.io/cuda-q-academic/interactive_widgets/ising_api_key_setup.html)  
* **Upload fails:** single RGB PNG/JPEG; shrink very large images  
* **Timeout:** retry once, then a smaller image or the browser playground  

References: [README.md](README.md).